In [ ]:
import sys, os, glob, shutil, time, json, gc
import numpy as np, pandas as pd
t0 = time.perf_counter()
def log(m): print(f"[{time.perf_counter()-t0:6.0f}с] {m}", flush=True)
os.makedirs("/kaggle/working/src", exist_ok=True)
os.makedirs("/kaggle/working/models", exist_ok=True)
code = os.path.dirname(glob.glob("/kaggle/input/**/pair_features.py", recursive=True)[0])
for p in glob.glob(code + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
for p in glob.glob(code + "/*.json"): shutil.copy(p, "/kaggle/working/models/")
# Версии из отправленного архива кладутся поверх: именно на них обучена структурная модель.
struct = os.path.dirname(glob.glob("/kaggle/input/**/pair_boost_hybrid.npz", recursive=True)[0])
for p in glob.glob(struct + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
for p in glob.glob(struct + "/*.npz"): shutil.copy(p, "/kaggle/working/models/")
open("/kaggle/working/src/__init__.py", "a").close()
os.chdir("/kaggle/working"); sys.path.insert(0, "/kaggle/working")
from src.pair_features import build_matrix, feature_names
from src.measure_features import measures, compare_measures, MEASURE_FEATURES
from src.features import extract_model_features
from src.model import BoostedPairModel
from src.export_boost import export, save, predict_proba
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score
from scipy.stats import rankdata, spearmanr

fold = os.path.dirname(glob.glob("/kaggle/input/**/llm_valid_pairs.parquet", recursive=True)[0])
sc = os.path.dirname(glob.glob("/kaggle/input/**/ce_relaxed.npy", recursive=True)[0])
pairs = pd.read_parquet(fold + "/llm_valid_pairs.parquet")
items = pd.read_parquet(fold + "/llm_valid_items.parquet")
y = (pairs["target"].to_numpy() > 0).astype(np.int8)
cat = pairs["id1"].map(dict(zip(items.id, items.category.astype(str)))).astype(str).to_numpy()
known = sorted(set(items.category.astype(str)))
log(f"пар {len(pairs):,}, доля+ {y.mean():.3f}")

names = list(feature_names(False))
X = np.zeros((len(pairs), len(names)), dtype=np.float32)
for c in known:
    rows = np.flatnonzero(cat == c)
    if not len(rows): continue
    sub = items[items.category.astype(str) == c].reset_index(drop=True)
    X[rows] = build_matrix(sub, pairs.iloc[rows].reset_index(drop=True), known, with_neighbours=False)
    del sub; gc.collect()
log("парные признаки готовы")

t = time.perf_counter()
legacy = extract_model_features(pairs[["id1", "id2"]], items)
primary = BoostedPairModel("models/pair_boost_hybrid.npz").predict_probability(legacy, cat)
aux = BoostedPairModel("models/pair_boost_hybrid_aux.npz").predict_probability(legacy, cat)
structural = (0.8 * primary + 0.2 * aux).astype(np.float32)
del legacy; gc.collect()
log(f"структурная модель за {time.perf_counter()-t:.0f}с")

M = {int(i): measures(a) for i, a in zip(items.id, items.attributes)}
MX = np.array([[r[n] for n in MEASURE_FEATURES] for r in
               (compare_measures(M[x], M[z]) for x, z in zip(pairs.id1, pairs.id2))], dtype=np.float32)
CE = {n: np.load(f"{sc}/{n}.npy").astype(np.float32) for n in ("ce_relaxed", "ce_combo")}
codes = np.array([known.index(c) if c in known else -1 for c in cat], dtype=np.float32)

BASE_COLS = names + list(MEASURE_FEATURES) + ["ce_relaxed", "ce_combo", "category_code"]
BASE = np.column_stack([X, MX, CE["ce_relaxed"], CE["ce_combo"], codes]).astype(np.float64)
WITH_COLS = names + list(MEASURE_FEATURES) + ["ce_relaxed", "ce_combo", "structural", "category_code"]
WITH = np.column_stack([X, MX, CE["ce_relaxed"], CE["ce_combo"], structural, codes]).astype(np.float64)

# Скоры новых моделей: контрастная лежит датасетом, обученная на трудных негативах —
# в выходе своего ядра. Качать её сюда нельзя, там 700 МБ весов, поэтому считаем на месте.
bi_dir = os.path.dirname(glob.glob("/kaggle/input/**/ce_bi.npy", recursive=True)[0])
BI = np.load(f"{bi_dir}/ce_bi.npy").astype(np.float32)
hn_hits = glob.glob("/kaggle/input/**/ce_hardneg/llm_ood_scores.npy", recursive=True)
HN_FULL = np.load(hn_hits[0]).astype(np.float32) if hn_hits else None
log(f"контрастная: {BI.shape}, негативы: {'нет' if HN_FULL is None else HN_FULL.shape}")

# Скоры негативов посчитаны на всех 200 тысячах пар фолда, а мы работаем на 156 тысячах —
# приводим по тем же парам, что и остальные модели.
full = pd.read_parquet(glob.glob("/kaggle/input/**/llm_valid.parquet", recursive=True)[0]) \
    if glob.glob("/kaggle/input/**/llm_valid.parquet", recursive=True) else None
if HN_FULL is not None and len(HN_FULL) != len(pairs):
    src_pairs = pd.read_parquet(os.path.dirname(hn_hits[0]) + "/llm_ood_pairs.parquet")
    pos = pd.Series(np.arange(len(src_pairs)),
                    index=pd.MultiIndex.from_arrays([src_pairs.id1, src_pairs.id2]))
    take = pos.reindex(pd.MultiIndex.from_arrays([pairs.id1, pairs.id2])).to_numpy()
    log(f"выравнивание по парам: найдено {np.isfinite(take).mean():.1%}")
    HN = HN_FULL[take.astype(int)]
else:
    HN = HN_FULL

CE = {n: np.load(f"{sc}/{n}.npy").astype(np.float32) for n in ("ce_relaxed", "ce_combo")}
codes = np.array([known.index(c) if c in known else -1 for c in cat], dtype=np.float32)
masks = {c: cat == c for c in np.unique(cat)}
def rk(s):
    o = np.empty(len(s), np.float32)
    for m in masks.values(): o[m] = rankdata(s[m]) / m.sum()
    return o
def macro(s, rate=0.111, seeds=8):
    vals = []
    for seed in range(seeds):
        rng = np.random.default_rng(seed); per = []
        for m in masks.values():
            rows = np.flatnonzero(m); pos_, neg = rows[y[rows]==1], rows[y[rows]==0]
            keep = min(len(pos_), max(5, int(round(rate/(1-rate)*len(neg)))))
            ch = np.concatenate([rng.choice(pos_, keep, replace=False), neg])
            per.append(average_precision_score(y[ch], s[ch]))
        vals.append(np.mean(per))
    return float(np.mean(vals)), float(np.std(vals))

if HN is not None:
    log(f"негативы сами по себе: {macro(rk(HN))[0]:.6f}")
    for n in ("ce_relaxed", "ce_combo"):
        log(f"  спирмен с {n}: {spearmanr(CE[n], HN).statistic:+.3f}")
log(f"контрастная сама по себе: {macro(rk(BI))[0]:.6f}")

PARAMS = dict(max_iter=500, max_leaf_nodes=63, learning_rate=0.06, l2_regularization=1.0,
              early_stopping=False, random_state=0)
half = np.random.default_rng(5).permutation(len(y)) % 2
def run(extra, tag):
    mat = np.column_stack([X, MX, CE["ce_relaxed"], CE["ce_combo"]] + extra +
                          [structural, codes]).astype(np.float32)
    p = np.zeros(len(y))
    for h in (0, 1):
        tr, te = half != h, half == h
        p[te] = HistGradientBoostingClassifier(**PARAMS).fit(mat[tr], y[tr]).predict_proba(mat[te])[:, 1]
    mu, sd = macro(rk(p))
    log(f"  {tag:<34} {mu:.6f} ± {sd:.6f}  ({mat.shape[1]} столбцов)")
    return mu

run([], "два энкодера")
run([BI], "+ контрастная")
if HN is not None:
    run([HN], "+ негативы")
    run([BI, HN], "+ обе")
